# Phase 0.7 — Stage 3: fine-tune pi0.5 on the portable-rig dataset

One of **three notebooks run in parallel**, one per policy, for the Q2 policy-transfer question of
[08 §A.1](08_data_quality_research.md): *do the same data properties hold across ACT, SmolVLA and π0.5?*
This notebook trains **pi0.5** only.

| | |
|---|---|
| Dataset | `..._20261003_211912_trimmed` — 50 episodes, idle-trimmed by [Stage 2](colab_phase07_prep_trim.ipynb) |
| GPU | **A100 40 GB (required)** |
| Expected runtime | **~6-12 h** — Part B prints the real figure for your GPU |
| Output | `HALDijkstraaa/pi05_tray_cylinder_s1000_trim` (private), checkpointed to the Hub as it goes |
| Siblings | [ACT](colab_phase07_act.ipynb) · [SmolVLA](colab_phase07_smolvla.ipynb) · [π0.5](colab_phase07_pi05.ipynb) |

**π0.5** fine-tunes `lerobot/pi05_base` (3.3 B). The two decisions worth knowing before you start:

- **A full fine-tune is impossible on Colab.** It needs >70 GB (09 §D: 52.8 GB of optimiser states plus
  ~17 GB of activations) and the largest Colab offers is a 40 GB A100. This notebook uses
  **`train_expert_only=true`** — the VLM is frozen and only the action expert and projections train.
- **That choice is deliberate, not just a memory workaround.** It is the same adaptation regime SmolVLA's
  base config uses by default, which keeps the SmolVLA↔π0.5 comparison about model class rather than
  about adaptation method — the confound 09 §D flags for Q2. LoRA is available in Part D as an
  alternative, but it would *introduce* that confound rather than remove it. ACT trains from scratch, so
  the asymmetry against ACT is unavoidable and belongs in the write-up.

It also needs a **rename map**: the checkpoint declares `base_0_rgb` / `left_wrist_0_rgb` /
`right_wrist_0_rgb`. The third is left unfed and π0.5 pads it with a −1 image and a zero attention mask,
which is exactly how openpi serves a one-base-one-wrist arm.

## Held constant across all three notebooks

Everything that is not the policy, so the comparison means something: the **same trimmed dataset**, the
**same balanced holdout** `[9, 19, 29, 39, 49]`, **seed 1000**, both cameras, no image augmentation,
validation every 5000 steps, and one wandb project (`phase07-policy-transfer`) so the curves overlay.

Per-policy hyperparameters are each policy's own published recipe — that is the intended design
([08 §A.4](08_data_quality_research.md) holds hyperparameters fixed *after* one clean run, not identical
across policies). **Freeze whatever this run lands on and reuse it for the dial grid.**

## How to use

**Part A** once. **Part B** is a ~15 min smoke test that fails loudly rather than warning — paste its
output before starting the long run. **Part C** is one cell: run it and walk away. It is **restartable**,
so after a Colab disconnect you re-run that same cell and it resumes from the last Hub checkpoint.

> **Do not start before Stage 2 prints `READY`.** Every notebook here reads the trimmed dataset; Part A
> checks it exists and is tagged.

---
# Part A — Setup

## 1 — Runtime

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", GPU)
assert GPU, "No GPU. Runtime > Change runtime type > GPU (A100)."
if "A100" not in GPU and "H100" not in GPU:
    print("\n!! " + GPU + " is too small. pi0.5 weights alone are 6.6 GB in bf16 and the")
    print("   action-expert fine-tune needs ~25-30 GB. Runtime > Change runtime type > A100.")

## 2 — Install

Pinned to **v0.6.1** — the version that recorded the dataset, and the one Phase 0.6's Colab runs used.

> Note there is **no `act` extra** in v0.6.1 (pip accepts an unknown extra silently, which is why the
> Phase 0.6 notebook's `lerobot[act]` appeared to work). `dataset` + `training` is what ACT actually needs.

In [ ]:
!pip install -q "lerobot[pi,dataset,training]==0.6.1"
import lerobot
print("lerobot", lerobot.__version__)

> If `import lerobot` fails, restart the runtime (**Runtime > Restart session**) and re-run this cell.
> Nothing later imports lerobot into this kernel — every use goes through a subprocess.

## 3 — Authentication

**A Colab secret alone is not enough.** `huggingface_hub` resolves a token as OIDC → `HF_TOKEN` env var →
token file → Colab secret, and that last path calls `google.colab.userdata.get()`, which needs the
kernel's channel to the Colab frontend. `lerobot-train` is a **subprocess**, where that channel does not
exist — so with only a secret set, `whoami()` succeeds in the notebook while training authenticates as
nobody. These cells copy the token into the environment *and* the token file, then prove a subprocess can
use it. The run also creates a new model repo, which a fine-grained token scoped to existing repos cannot
do; that is asserted too.

In [ ]:
import os, sys, subprocess
from huggingface_hub import get_token, login, whoami

tok = os.environ.get("HF_TOKEN") or get_token()
if not tok:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        tok = None
if not tok:
    from huggingface_hub import notebook_login
    notebook_login(); tok = get_token()
assert tok, "no HF token - add a Colab secret named HF_TOKEN, or run notebook_login()"

os.environ["HF_TOKEN"] = tok                      # inherited by every subprocess
login(token=tok, add_to_git_credential=False)     # also writes ~/.cache/huggingface/token

r = subprocess.run([sys.executable, "-c",
                    "from huggingface_hub import whoami; print(whoami()['name'])"],
                   capture_output=True, text=True)
assert r.returncode == 0, "a subprocess still cannot authenticate:\n" + r.stderr[-800:]
print("HF ok - notebook:", whoami()["name"], "| subprocess:", r.stdout.strip())

auth = whoami().get("auth", {}).get("accessToken", {})
if auth.get("role") == "fineGrained":
    scopes = (auth.get("fineGrained") or {}).get("global", [])
    print("token: fine-grained | global scopes:", scopes)
    assert any("repo.write" in s for s in scopes), (
        "this token cannot CREATE the model repo this run pushes to. Use a token with global "
        "repo.write from https://huggingface.co/settings/tokens.")

In [ ]:
import os, sys, subprocess, wandb

key = os.environ.get("WANDB_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        key = None
if key:
    os.environ["WANDB_API_KEY"] = key; wandb.login(key=key)
else:
    wandb.login()                                  # key from https://wandb.ai/authorize
    os.environ["WANDB_API_KEY"] = wandb.api.api_key

r = subprocess.run([sys.executable, "-c", "import wandb; print(bool(wandb.api.api_key))"],
                   capture_output=True, text=True)
assert r.stdout.strip() == "True", "a subprocess cannot see the wandb key:\n" + r.stderr[-500:]
print("wandb ok in notebook and subprocess")

## 4 — Configuration


| Knob | Value | Why |
|---|---|---|
| Base | `lerobot/pi05_base` | The published checkpoint |
| Batch / steps | 16 / 30,000 | 480k samples ≈ 16 epochs — the same sample budget as the ACT run |
| Precision | bf16 + gradient checkpointing | Required to fit 40 GB; the LeRobot π0.5 recipe uses both |
| Adaptation | `train_expert_only=true`, vision encoder frozen | Matches SmolVLA's regime (see above). Full fine-tune needs >70 GB |
| LR / schedule | 2.5e-5, cosine with 1k warmup | Policy preset. `scheduler_decay_steps` pinned to the real step count |
| Images | 224×224, both cameras + one padded empty | Base config; the unfed third camera gets a zero mask |

In [ ]:
import collections, json, os, re, subprocess, sys, threading, time
HF_USER = "HALDijkstraaa"
DATASET = f"{HF_USER}/so101_tray_cylinder_portable_20261003_211912_trimmed"
PROJECT = "phase07-policy-transfer"
SEED    = 1000

# Held out for validation loss. Episodes 0-44 are 9 tray-spot blocks of 5 CONSECUTIVE episodes and
# 45-49 are off-grid, so lerobot's default "last ceil(n*eval_split) episodes" split would hold out
# exactly the 5 off-grid episodes - removing the only unmarked placements from training. The split
# slices the tail of --dataset.episodes IN THE ORDER GIVEN, so putting the holdout last reorders it
# into a balanced one: the last episode of blocks 2, 4, 6, 8 plus one off-grid.
HOLDOUT  = [9, 19, 29, 39, 49]
EPISODES = [e for e in range(50) if e not in HOLDOUT] + HOLDOUT

EVAL_SPLIT, EVAL_STEPS, LOG_FREQ = 0.1, 5000, 200

# ---- pi0.5-specific ----------------------------------------------------------------------------
JOB   = "pi05_tray_cylinder_s1000_trim"
# The published recipe. These two never change; BATCH/STEPS below are what the run actually uses, and
# Part B derives them from these so re-running it is idempotent rather than compounding.
BASE_BATCH, BASE_STEPS = 16, 30000
BATCH, STEPS = BASE_BATCH, BASE_STEPS
SAVE_FREQ = 5000
BASE_MODEL = "lerobot/pi05_base"

# The pretrained checkpoint declares its own camera names; the batch is renamed to match.
RENAME = {
    "observation.images.top": "observation.images.base_0_rgb",
    "observation.images.wrist": "observation.images.left_wrist_0_rgb"
}

NUM_WORKERS = min(8, os.cpu_count() or 2)


def train_cmd(job=None, steps=None, batch=None, extra=()):
    """Build the lerobot-train argv.

    Every default is resolved from the globals at CALL time, not bound at definition time, so a
    batch step-down in Part B reaches a later bare train_cmd() call. Checkpoints go to the Hub every
    SAVE_FREQ steps, so a disconnect costs at most that many.
    """
    job = JOB if job is None else job
    steps = STEPS if steps is None else steps
    batch = BATCH if batch is None else batch
    return ["lerobot-train",
        f"--dataset.repo_id={DATASET}", f"--dataset.episodes={EPISODES}",
        f"--dataset.eval_split={EVAL_SPLIT}", f"--eval_steps={EVAL_STEPS}",
        f"--policy.path={BASE_MODEL}",
        f"--rename_map={json.dumps(RENAME)}",
        "--policy.dtype=bfloat16",
        "--policy.gradient_checkpointing=true",
        "--policy.train_expert_only=true",
        "--policy.freeze_vision_encoder=true",

        "--policy.device=cuda",
        "--policy.push_to_hub=true", f"--policy.repo_id={HF_USER}/{job}", "--policy.private=true",
        "--save_checkpoint_to_hub=true",
        # Pin the cosine decay to the real step count so the schedule always completes,
        # whatever batch the smoke test settles on (the preset default is 30k).
        f"--policy.scheduler_decay_steps={steps}",
        f"--batch_size={batch}", f"--steps={steps}", f"--num_workers={NUM_WORKERS}",
        f"--seed={SEED}", f"--save_freq={SAVE_FREQ}", f"--log_freq={LOG_FREQ}",
        f"--output_dir=/content/outputs/{job}", f"--job_name={job}",
        "--wandb.enable=true", f"--wandb.project={PROJECT}", *extra]


KEEP = ("step:", "eval_loss", "Train/eval split", "num_learnable_params", "input_features",
        "Traceback", "Error", "error:", "Track this run", "End of training", "out of memory")


def gpu_watch(stop, peak):
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip().splitlines()
        if out:
            peak[0] = max(peak[0], int(out[0]))
        time.sleep(5)


def run(cmd, quiet=True):
    """Stream a run, dropping tqdm spam, and record peak GPU memory alongside throughput."""
    t0, tail = time.time(), collections.deque(maxlen=80)
    peak, stop = [0], threading.Event()
    threading.Thread(target=gpu_watch, args=(stop, peak), daemon=True).start()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        tail.append(line)
        if (not quiet) or any(k in line for k in KEEP) or "wandb.ai" in line:
            sys.stdout.write(line); sys.stdout.flush()
    p.wait(); stop.set()
    dt, text = (time.time() - t0) / 3600, "".join(tail)
    print(f"[exit {p.returncode} after {dt:.2f} h | peak GPU {peak[0]/1024:.1f} GiB]")
    if p.returncode != 0:
        print("--- last lines ---"); print(text[-2500:])
    return {"code": p.returncode, "hours": dt, "peak_gib": peak[0] / 1024, "tail": text,
            "oom": "out of memory" in text.lower() or "CUDA out of memory" in text}


print("holdout      :", HOLDOUT)
print("episode order:", EPISODES[:6], "...", EPISODES[-6:])
print("workers      :", NUM_WORKERS)

## 5 — Preflight

Three things that each cost minutes now and hours later: the dataset exists and carries its
codebase-version git tag; the holdout reorder is well formed; and the camera keys line up with what the
policy expects. None of this needs a GPU.

In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
checks = []

ok_ds = api.repo_exists(DATASET, repo_type="dataset")
checks.append(("trimmed dataset exists", ok_ds))
assert ok_ds, f"{DATASET} not found. Run colab_phase07_prep_trim.ipynb (Stage 2) first."

info = json.loads(open(hf_hub_download(DATASET, "meta/info.json", repo_type="dataset")).read())
tags = [t.name for t in api.list_repo_refs(DATASET, repo_type="dataset").tags]
# lerobot resolves a dataset at the revision named by its codebase_version; without the tag, training
# dies several minutes in with "Your dataset must be tagged with a codebase version".
checks.append((f"codebase-version tag {info['codebase_version']}", info["codebase_version"] in tags))
checks.append(("50 episodes", info["total_episodes"] == 50))
checks.append(("idle-trimmed (fewer frames than the source's 35,141)", info["total_frames"] < 35141))
print(f"dataset: {info['total_episodes']} episodes, {info['total_frames']} frames, "
      f"{info['fps']} fps, tags={tags}")

checks.append(("holdout reorder is a permutation of 0-49", sorted(EPISODES) == list(range(50))))
checks.append(("holdout is the tail of the episode list", EPISODES[-len(HOLDOUT):] == HOLDOUT))

ds_cams = {k for k in info["features"] if k.startswith("observation.images.")}
# The decisive check for a pretrained policy: after renaming, the batch's camera keys must be a subset
# of what the checkpoint declares. lerobot accepts a subset in either direction, and anything the policy
# declares but the batch does not supply is padded internally.
pol_cfg = json.loads(open(hf_hub_download(BASE_MODEL, "config.json")).read())
pol_cams = {k for k, v in pol_cfg["input_features"].items() if v["type"] == "VISUAL"}
renamed = {RENAME.get(k, k) for k in ds_cams}
print("\npolicy declares :", sorted(pol_cams))
print("dataset supplies:", sorted(ds_cams))
print("after rename    :", sorted(renamed))
print("left unfed      :", sorted(pol_cams - renamed), "(padded internally with a zero mask)")
checks += [
    ("rename map keys exist in the dataset", set(RENAME) <= ds_cams),
    ("renamed keys are a subset of the policy's", renamed <= pol_cams),
]
for k, v in sorted(pol_cfg.items()):
    if k in ("freeze_vision_encoder", "train_expert_only", "optimizer_lr", "chunk_size",
             "n_action_steps", "dtype", "gradient_checkpointing", "resize_imgs_with_padding",
             "image_resolution"):
        print(f"  base config  {k:28s} = {v}")

print()
for n, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n}")
assert all(ok for _, ok in checks), "preflight failed - fix before running Part B"
print("\npreflight OK")

---
# Part B — Smoke test

One cell, ~15 minutes. Runs **300 real training steps** — the balanced split, periodic validation, wandb,
and a checkpoint pushed to the Hub — then checks every result and prints a readiness verdict plus the
projected runtime for the full run on *your* GPU.

If it hits CUDA OOM it **halves the batch and retries**, then scales the step count to hold
samples-seen fixed, so the published recipe's sample budget survives a smaller GPU. Whatever it settles
on is what Part C uses.

It fails loudly rather than warning, so Part C cannot start on a broken setup.

In [ ]:
from huggingface_hub import HfApi

SMOKE_REPO, N = f"{HF_USER}/pi05_smoke_delete_me", 300
batch, res = BATCH, None

while True:
    print(f"\n{'=' * 70}\n  smoke test at batch {batch}\n{'=' * 70}")
    subprocess.run(["rm", "-rf", "/content/outputs/smoke"], check=False)
    cmd = [c for c in train_cmd(job="smoke", steps=N, batch=batch)
           if not c.startswith(("--policy.repo_id", "--save_freq", "--eval_steps"))]
    cmd += [f"--policy.repo_id={SMOKE_REPO}", f"--eval_steps={N // 2}", f"--save_freq={N}"]
    res = run(cmd)

    if res["oom"] and batch > 1:
        print(f"\n!! OOM at batch {batch}. Halving and retrying.")
        print("   Steps will be scaled to keep samples-seen constant, so the recipe's sample budget")
        print("   is preserved even though the batch is smaller.")
        batch //= 2
        continue
    break

text = res["tail"]
rates = [float(x) for x in re.findall(r"([\d.]+)\s*step/s", text)]
checks = [
    ("training completed",                 res["code"] == 0),
    ("balanced split 45 train / 5 eval",   "45 train, 5 eval" in text),
    ("validation ran",                     "eval_loss=" in text),
    ("wandb connected",                    "wandb.ai" in text),
]
try:
    files = HfApi().list_repo_files(SMOKE_REPO, repo_type="model")
    checks.append(("checkpoint uploaded to the Hub", any("checkpoints/" in f for f in files)))
except Exception as e:
    checks.append((f"checkpoint uploaded to the Hub ({type(e).__name__})", False))

print()
for n_, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n_}")

losses = [float(x) for x in re.findall(r"loss:\s*([\d.]+)", text)]
if len(losses) >= 2:
    checks.append((f"loss fell over {len(losses)} logged points "
                   f"({losses[0]:.3f} -> {losses[-1]:.3f})", losses[-1] < losses[0]))

BATCH = batch
# Derived from the published recipe, not from the current STEPS, so re-running this cell after a
# step-down does not scale the step count a second time.
STEPS = int(round(BASE_STEPS * BASE_BATCH / BATCH))
rate = sorted(rates)[len(rates) // 2] if rates else None

print(f"\n{'=' * 70}")
print(f"  GPU            : {GPU}")
print(f"  peak memory    : {res['peak_gib']:.1f} GiB")
if rate:
    hours = STEPS / rate / 3600
    print(f"  throughput     : {rate:.2f} step/s  ({rate * BATCH:.0f} samples/s)")
    print(f"  batch          : {BATCH}"
          + ("" if BATCH == BASE_BATCH else f"   (stepped down from {BASE_BATCH})"))
    print(f"  steps          : {STEPS:,}   ({STEPS * BATCH / 1000:.0f}k samples)")
    print(f"  PROJECTED RUN  : {hours:.1f} h  (+ validation, roughly 5%)")
    if hours > 12:
        print("\n  !! Over 12 h. Colab will likely disconnect at least once - Part C resumes from")
        print("     the last Hub checkpoint, so that is survivable, but consider either a bigger GPU")
        print("     or setting STEPS by hand to accept a smaller sample budget.")
else:
    print("  throughput     : could not parse step/s from the log")

SMOKE_OK = all(ok for _, ok in checks) and rate is not None
print(f"\n  READY: {SMOKE_OK}")
print("=" * 70)
if SMOKE_OK:
    print("\nPaste the block above, then run Part C.")
    print(f"Clean up when convenient:  HfApi().delete_repo('{SMOKE_REPO}', repo_type='model')")
else:
    raise RuntimeError("smoke test failed - see the FAIL lines above. Do not start Part C.")

---
# Part C — Run it

**One cell. Run it and walk away.**

**Restartable.** If Colab disconnects, re-run this same cell. It first reads the run's Hub repo: a final
`model.safetensors` means finished (skip), checkpoints without one mean part-done (resume from the last,
losing at most 5,000 steps), neither means start. So a disconnect never repeats completed
work, and you never have to work out where it got to.

If you reconnected and `SMOKE_OK` is unset, that is expected — re-running Part B costs 15 minutes and
proves nothing new, so this cell warns rather than blocking.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

_api = HfApi()

if not globals().get("SMOKE_OK"):
    print("!! Part B has not run in this kernel (SMOKE_OK is unset).")
    print("   Expected after a reconnect: this cell is restartable and re-checks the Hub itself.")
    print("   On a fresh setup, run Part B first - it must print READY: True.\n")


def run_state(job):
    """Where this run stands, read from its Hub repo: finished, part-done, or not started."""
    try:
        files = _api.list_repo_files(f"{HF_USER}/{job}", repo_type="model")
    except Exception:
        return "new", None
    if "model.safetensors" in files:            # the final push only happens at the very end
        return "done", None
    steps = sorted({f.split("/")[1] for f in files
                    if f.startswith("checkpoints/") and f.split("/")[1].isdigit()})
    return ("resume", steps[-1]) if steps else ("new", None)

t0 = time.time()
where, last = run_state(JOB)

if where == "done":
    print(f"already finished on the Hub - nothing to do: https://huggingface.co/{HF_USER}/{JOB}")
    res = {"code": 0, "hours": 0.0}
elif where == "resume":
    # Prefer the local checkpoint if this VM still has it; otherwise pull from the Hub. The step
    # counter, optimizer state, episode order, holdout and seed all come back from the checkpoint -
    # only --steps is read from the CLI.
    local = Path(f"/content/outputs/{JOB}/checkpoints/last/pretrained_model/train_config.json")
    src = str(local) if local.exists() else f"{HF_USER}/{JOB}"
    print(f"resuming from step {last} ({'local' if local.exists() else 'Hub'})")
    cmd = ["lerobot-train", f"--config_path={src}", "--resume=true", f"--steps={STEPS}"]
    if not local.exists():
        cmd.append(f"--output_dir=/content/outputs/{JOB}_resumed")
    res = run(cmd)
else:
    print(f"starting fresh: {STEPS:,} steps at batch {BATCH}")
    res = run(train_cmd(steps=STEPS, batch=BATCH))

status = "ok" if res["code"] == 0 else f"FAILED (exit {res['code']})"
print(f"\n{'=' * 70}")
print(f"  {JOB}")
print(f"  status : {status}")
print(f"  time   : {res['hours']:.2f} h")
print(f"  model  : https://huggingface.co/{HF_USER}/{JOB}")
print(f"  wandb  : project {PROJECT}, run {JOB}")
print("=" * 70)
if res["code"] != 0:
    print("\nRe-run this cell to resume from the last Hub checkpoint, or see Part D.")
else:
    print("\nPaste this summary back. Then run the other two policies' notebooks if they are not")
    print("already going, and record the result in Details/phase07_training.md.")

---
# Part D — Recovery

Only if something in Part C failed.

**Resume by hand.** Part C does this automatically, but if you want to force it:

```python
run(["lerobot-train", f"--config_path={HF_USER}/{JOB}", "--resume=true", f"--steps={STEPS}",
     "--output_dir=/content/outputs/resumed"])
```

**Start over.** Delete the model repo first, or `run_state` will see the old checkpoints and resume:

```python
HfApi().delete_repo(f"{HF_USER}/{JOB}", repo_type="model")   # irreversible
```

**OOM partway through a long run.** The batch that passed the smoke test should hold, but if a later
phase OOMs, drop the batch and resume — the optimizer state carries over:

```python
res = run(["lerobot-train", f"--config_path={HF_USER}/{JOB}", "--resume=true",
           f"--steps={STEPS}", f"--batch_size={BATCH // 2}"])
```

**It trained but you are not sure it trained on the right thing.** Check the run log for
`Train/eval split: 45 train, 5 eval` and that the episode order put `[9, 19, 29, 39, 49]` last. Both are
printed near the top of Part C's output and are also in the wandb config.

**Switching to LoRA instead of `train_expert_only`.** Only if you decide the frozen-VLM regime is too
restrictive — note that it *introduces* the adaptation-method confound against SmolVLA that the current
setting avoids (09 §D), so say so in the write-up if you do.

```python
!pip install -q "lerobot[peft]==0.6.1"
res = run(train_cmd(job=JOB + "_lora", extra=("--peft.method_type=LORA", "--peft.r=16",
                                              "--peft.lora_alpha=32")))
```

lerobot's π0.5 LoRA defaults target the action expert's q/v projections plus the state/action
projections — PaliGemma itself is left untouched, so it is close in spirit to `train_expert_only`
anyway.

---

### When all three are done

Collect the three summaries into `Details/phase07_training.md` — held-fixed table, per-policy recipe,
measured runtime and peak memory, final `eval_loss`, and the pre-registered predictions for the robot
rollouts. The three runs share the wandb project `phase07-policy-transfer`, so the eval-loss curves
overlay directly — but note that **eval loss is an action-prediction L1 and correlates only weakly with
task success** ([phase06 §2](Details/phase06_camera_ablation_training.md)); it is a divergence sanity
check, not the answer. The answer is rollouts on the robot.